In [1]:
import math
import torch
import torch.nn as nn
from torch.utils.data import Dataset


class DecoderWithEmbeddingDataset(Dataset):
    def __init__(self, embeddings, strings, token2id, max_len=30):
        self.embeddings = torch.tensor(embeddings).float()
        self.strings    = strings
        self.token2id   = token2id
        self.max_len    = max_len
        self.pad_id     = token2id["<PAD>"]

    def __len__(self):
        return len(self.strings)

    def __getitem__(self, idx):
        z      = self.embeddings[idx]
        t2i    = self.token2id
        tokens = (
            [t2i["<BOS>"]]
            + [t2i.get(t, t2i["<UNK>"]) for t in self.strings[idx].split()]
            + [t2i["<EOS>"]]
        )
        if len(tokens) < self.max_len:
            tokens += [self.pad_id] * (self.max_len - len(tokens))
        else:
            tokens = tokens[: self.max_len]

        tokens    = torch.tensor(tokens)
        input_ids = tokens[:-1].clone()
        labels    = tokens[1:].clone()
        labels[labels == self.pad_id] = -100
        return z, input_ids, labels


class SinusoidalPE(nn.Module):
    def __init__(self, d_model, max_len=512, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe  = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return self.dropout(x + self.pe[:, : x.size(1)])


class DecoderLayer(nn.Module):
    def __init__(self, d_model=256, n_heads=8, ffn_dim=1024, dropout=0.1):
        super().__init__()
        self.self_attn  = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.norm1      = nn.LayerNorm(d_model)
        self.cross_attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.norm2      = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, ffn_dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(ffn_dim, d_model), nn.Dropout(dropout),
        )
        self.norm3 = nn.LayerNorm(d_model)

    def forward(self, x, memory, causal_mask):
        r = x; x = self.norm1(x)
        x, _ = self.self_attn(x, x, x, attn_mask=causal_mask, is_causal=True)
        x = r + x
        r = x; x = self.norm2(x)
        x, _ = self.cross_attn(x, memory, memory)
        x = r + x
        r = x; x = self.norm3(x)
        x = self.ffn(x)
        return r + x


class TransformerJEPADecoder(nn.Module):
    def __init__(self, latent_dim=128, d_model=256, n_heads=8, n_layers=4,
                 ffn_dim=1024, vocab_size=27, max_len=30, dropout=0.1,
                 z_seq_len=4, bos_id=1):
        super().__init__()
        self.max_len    = max_len
        self.vocab_size = vocab_size
        self.bos_id     = bos_id
        self.d_model    = d_model

        self.z_proj = nn.Sequential(
            nn.Linear(latent_dim, d_model * z_seq_len),
            nn.GELU(),
            nn.Unflatten(-1, (z_seq_len, d_model)),
        )
        self.embedding  = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pos_enc    = SinusoidalPE(d_model, max_len=max_len + 2, dropout=dropout)
        self.layers     = nn.ModuleList([DecoderLayer(d_model, n_heads, ffn_dim, dropout) for _ in range(n_layers)])
        self.final_norm = nn.LayerNorm(d_model)
        self.fc_out     = nn.Linear(d_model, vocab_size)
        self.fc_out.weight = self.embedding.weight  # weight tying

    @staticmethod
    def _causal_mask(seq_len, device):
        return torch.triu(torch.full((seq_len, seq_len), float("-inf"), device=device), diagonal=1)

    def forward(self, z, target_tokens=None, teacher_forcing_ratio=0.5):
        B, device = z.size(0), z.device
        memory = self.z_proj(z)  # (B, z_seq_len, d_model)

        # pure teacher forcing — single parallel pass
        if target_tokens is not None and (teacher_forcing_ratio >= 1.0 or not self.training):
            T    = target_tokens.size(1)
            x    = self.pos_enc(self.embedding(target_tokens) * math.sqrt(self.d_model))
            mask = self._causal_mask(T, device)
            for layer in self.layers:
                x = layer(x, memory, mask)
            return self.fc_out(self.final_norm(x))

        # mixed / autoregressive — step by step
        seq        = torch.full((B, 1), self.bos_id, device=device, dtype=torch.long)
        all_logits = []
        T = target_tokens.size(1) if target_tokens is not None else self.max_len

        for t in range(T):
            x    = self.pos_enc(self.embedding(seq) * math.sqrt(self.d_model))
            mask = self._causal_mask(seq.size(1), device)
            for layer in self.layers:
                x = layer(x, memory, mask)
            logits = self.fc_out(self.final_norm(x))
            all_logits.append(logits[:, -1:])

            if target_tokens is not None and torch.rand(1).item() < teacher_forcing_ratio:
                next_tok = target_tokens[:, t].unsqueeze(1)
            else:
                next_tok = logits[:, -1].argmax(-1, keepdim=True)
            seq = torch.cat([seq, next_tok], dim=1)

        return torch.cat(all_logits, dim=1)

In [2]:
combined_matrix = torch.load("/content/drive/MyDrive/symbolic_data/final_data_and_model_lm_jepa_3march/equation_embedding_library_for_retrieval.pt", weights_only=False)

In [13]:
from tqdm import tqdm
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.amp import GradScaler, autocast
MAX_LEN = 30
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
def train_decoder(decoder, embeddings, strings, token2id, pad_id, max_len, epochs=50, batch_size=256):
    decoder.to(device)
    optimizer = torch.optim.AdamW(decoder.parameters(), lr=1e-3, weight_decay=1e-2)
    criterion = nn.CrossEntropyLoss(ignore_index=-100)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    dataset = DecoderWithEmbeddingDataset(embeddings, strings, token2id, max_len=max_len)
    loader  = DataLoader(dataset, batch_size=batch_size, num_workers=2,
                         pin_memory=True, shuffle=True, persistent_workers=True)

    scaler = torch.amp.GradScaler('cuda')

    for epoch in range(epochs):
        decoder.train()
        total_loss = 0

        pbar = tqdm(loader, desc=f"Epoch {epoch+1}/{epochs}")
        for z_batch, input_ids, labels in pbar:
            z_batch   = z_batch.to(device, non_blocking=True)
            input_ids = input_ids.to(device, non_blocking=True)
            labels    = labels.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast('cuda'):
                # teacher_forcing_ratio=1.0 forces the fast parallel path —
                # no Python for-loop, entire sequence in one pass through all layers
                logits = decoder(z_batch, target_tokens=input_ids, teacher_forcing_ratio=1.0)
                loss   = criterion(logits.reshape(-1, decoder.vocab_size), labels.reshape(-1))

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(decoder.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()

            total_loss += loss.item()
            pbar.set_postfix(loss=f"{loss.item():.4f}")

        scheduler.step()
        print(f"Epoch {epoch+1} — Avg Loss: {total_loss/len(loader):.4f}")

    torch.save(decoder.state_dict(), "/content/drive/MyDrive/symbolic_data/final_jepa_model/jepa_decoder_3_march.pt")


In [14]:
combined_matrix = torch.load("/content/drive/MyDrive/symbolic_data/final_data_and_model_lm_jepa_3march/equation_embedding_library_for_retrieval.pt", weights_only=False)
strings = pd.read_csv("/content/drive/MyDrive/symbolic_data/final_data_and_model_lm_jepa_3march/feynman_50k_unified_library_catalog.csv")
equations = strings['equation_string'].to_list()
PAD_ID, MASK_ID = TOKEN2ID["<PAD>"], TOKEN2ID["<MASK>"]

train_decoder(TransformerJEPADecoder(), combined_matrix, equations, TOKEN2ID, PAD_ID, MAX_LEN)

Epoch 1/50: 100%|██████████| 196/196 [00:09<00:00, 20.82it/s, loss=2.5219]


Epoch 1 — Avg Loss: 13.6183


Epoch 2/50: 100%|██████████| 196/196 [00:09<00:00, 21.40it/s, loss=2.3285]


Epoch 2 — Avg Loss: 2.4084


Epoch 3/50: 100%|██████████| 196/196 [00:09<00:00, 20.17it/s, loss=2.1753]


...

Epoch 48 — Avg Loss: 1.2106


Epoch 49/50: 100%|██████████| 196/196 [00:09<00:00, 21.37it/s, loss=1.1862]


Epoch 49 — Avg Loss: 1.2085


Epoch 50/50: 100%|██████████| 196/196 [00:09<00:00, 21.58it/s, loss=1.1913]


Epoch 50 — Avg Loss: 1.2097


In [6]:
import pandas as pd

checkpoint = torch.load(
    "/content/drive/MyDrive/symbolic_data/final_jepa_model/tiny_target_encoder_feynman_21pct (1).pt",
    map_location="cuda" # Load to CPU first to avoid device-side assert during initial load
)

# Update global TOKEN2ID, ID2TOKEN, and VOCAB_SIZE from the loaded checkpoint
TOKEN2ID = checkpoint['vocab']
ID2TOKEN = {v: k for k, v in TOKEN2ID.items()}
VOCAB_SIZE = len(TOKEN2ID)


In [ ]:
@torch.no_grad()
def test_decode(decoder, embedding, id2token):
    decoder.eval()
    z = torch.tensor(embedding).float().unsqueeze(0).to(device)
    logits = decoder(z, target_tokens=None, teacher_forcing_ratio=0.0)
    ids = logits.argmax(dim=-1).squeeze(0).tolist()
    tokens = [id2token[i] for i in ids]
    # stop at EOS
    if TOKEN2ID["<EOS>"] in ids:
        tokens = tokens[:ids.index(TOKEN2ID["<EOS>"])]
    return " ".join(tokens)

# Test on a few examples
decoder = JEPADecoder(vocab_size=VOCAB_SIZE, max_len=MAX_LEN)
decoder.load_state_dict(torch.load("/content/drive/MyDrive/Colab Notebooks/symbolic_data/final_jepa_model/jepa_decoder_3_march_updraded.pt"))
decoder.to(device)

ID2TOKEN = {v: k for k, v in TOKEN2ID.items()}
for i in range(5):
    pred = test_decode(decoder, combined_matrix[i], ID2TOKEN)
    print(f"Target: {equations[i]}")
    print(f"Pred:   {pred}\n")

In [43]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm


# ─────────────────────────────────────────────────────────────────────────────
# Dataset
# ─────────────────────────────────────────────────────────────────────────────

class DecoderWithEmbeddingDataset(Dataset):
    def __init__(self, embeddings, strings, token2id, max_len=30):
        # normalize embeddings to zero mean, unit std — critical for cross-attn
        emb = torch.tensor(embeddings).float()
        self.embeddings = (emb - emb.mean(0)) / (emb.std(0) + 1e-8)

        self.strings  = strings
        self.token2id = token2id
        self.max_len  = max_len
        self.pad_id   = token2id["<PAD>"]

    def __len__(self):
        return len(self.strings)

    def __getitem__(self, idx):
        z   = self.embeddings[idx]
        t2i = self.token2id
        tokens = (
            [t2i["<BOS>"]]
            + [t2i.get(t, t2i["<UNK>"]) for t in self.strings[idx].split()]
            + [t2i["<EOS>"]]
        )
        if len(tokens) < self.max_len:
            tokens += [self.pad_id] * (self.max_len - len(tokens))
        else:
            tokens = tokens[: self.max_len]

        tokens    = torch.tensor(tokens)
        input_ids = tokens[:-1].clone()
        labels    = tokens[1:].clone()

        # mask PAD AND everything after the first EOS
        eos_id = t2i["<EOS>"]
        eos_positions = (labels == eos_id).nonzero(as_tuple=True)[0]
        if len(eos_positions) > 0:
            first_eos = eos_positions[0].item()
            labels[first_eos + 1:] = -100   # keep the EOS itself, mask everything after

        labels[labels == self.pad_id] = -100
        return z, input_ids, labels

# ─────────────────────────────────────────────────────────────────────────────
# Architecture
# ─────────────────────────────────────────────────────────────────────────────

class SinusoidalPE(nn.Module):
    def __init__(self, d_model, max_len=512, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe  = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return self.dropout(x + self.pe[:, : x.size(1)])


class DecoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, ffn_dim, dropout):
        super().__init__()
        self.self_attn  = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.norm1      = nn.LayerNorm(d_model)
        self.cross_attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.norm2      = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, ffn_dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(ffn_dim, d_model), nn.Dropout(dropout),
        )
        self.norm3 = nn.LayerNorm(d_model)

    def forward(self, x, memory, causal_mask):
        # pre-LN masked self-attention
        r = x; x = self.norm1(x)
        x, _ = self.self_attn(x, x, x, attn_mask=causal_mask, is_causal=True)
        x = r + x
        # pre-LN cross-attention over latent memory
        r = x; x = self.norm2(x)
        x, _ = self.cross_attn(x, memory, memory)
        x = r + x
        # pre-LN FFN
        r = x; x = self.norm3(x)
        return r + self.ffn(x)


class TransformerJEPADecoder(nn.Module):
    """
    Upgrades vs previous version:
      - z_proj uses a deeper 2-layer MLP with LayerNorm for stable projection
      - learnable query tokens concatenated to z_proj memory (gives model
        fixed "slots" to write into, separate from the embedding signal)
      - label smoothing in the output (handled in training criterion)
      - final LayerNorm before output head
    """
    def __init__(
        self,
        latent_dim = 128,
        d_model    = 256,
        n_heads    = 8,
        n_layers   = 4,
        ffn_dim    = 1024,
        vocab_size = 27,
        max_len    = 30,
        dropout    = 0.1,
        z_seq_len  = 8,      # more memory slots vs previous 4
        bos_id     = 1,
    ):
        super().__init__()
        self.max_len    = max_len
        self.vocab_size = vocab_size
        self.bos_id     = bos_id
        self.d_model    = d_model

        # deeper z projection — 2-layer MLP + LayerNorm
        self.z_proj = nn.Sequential(
            nn.Linear(latent_dim, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model * z_seq_len),
            nn.Unflatten(-1, (z_seq_len, d_model)),
        )

        # learnable "register" tokens appended to memory
        # lets the model write latent info into fixed slots without
        # competing with the positional structure of z_proj
        self.register_tokens = nn.Parameter(torch.randn(1, 4, d_model) * 0.02)

        self.embedding  = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pos_enc    = SinusoidalPE(d_model, max_len=max_len + 2, dropout=dropout)
        self.layers     = nn.ModuleList([
            DecoderLayer(d_model, n_heads, ffn_dim, dropout) for _ in range(n_layers)
        ])
        self.final_norm = nn.LayerNorm(d_model)
        self.fc_out     = nn.Linear(d_model, vocab_size, bias=False)
        self.fc_out.weight = self.embedding.weight   # weight tying

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, std=0.02)

    @staticmethod
    def _causal_mask(seq_len, device):
        return torch.triu(
            torch.full((seq_len, seq_len), float("-inf"), device=device), diagonal=1
        )

    def _build_memory(self, z):
        B = z.size(0)
        z_mem  = self.z_proj(z)                                        # (B, z_seq_len, d_model)
        regs   = self.register_tokens.expand(B, -1, -1)                # (B, 4, d_model)
        return torch.cat([z_mem, regs], dim=1)                         # (B, z_seq_len+4, d_model)

    def forward(self, z, target_tokens=None, teacher_forcing_ratio=1.0):
        B, device = z.size(0), z.device
        memory    = self._build_memory(z)

        if target_tokens is not None:
            T    = target_tokens.size(1)
            x    = self.pos_enc(self.embedding(target_tokens) * math.sqrt(self.d_model))
            mask = self._causal_mask(T, device)
            for layer in self.layers:
                x = layer(x, memory, mask)
            return self.fc_out(self.final_norm(x))

        # autoregressive inference only
        seq, all_logits = torch.full((B, 1), self.bos_id, device=device, dtype=torch.long), []
        for _ in range(self.max_len):
            x    = self.pos_enc(self.embedding(seq) * math.sqrt(self.d_model))
            mask = self._causal_mask(seq.size(1), device)
            for layer in self.layers:
                x = layer(x, memory, mask)
            logits = self.fc_out(self.final_norm(x))
            all_logits.append(logits[:, -1:])
            next_tok = logits[:, -1].argmax(-1, keepdim=True)
            seq = torch.cat([seq, next_tok], dim=1)
        return torch.cat(all_logits, dim=1)


# ─────────────────────────────────────────────────────────────────────────────
# Training
# ─────────────────────────────────────────────────────────────────────────────

def train_decoder(
    decoder,
    embeddings,
    strings,
    token2id,
    pad_id,
    max_len,
    epochs     = 50,
    batch_size = 256,
    lr         = 3e-4,
    warmup_epochs = 5,
    save_path  = "/content/drive/MyDrive/symbolic_data/final_jepa_model/jepa_decoder_3_march.pt",
):
    decoder.to(device)

    dataset = DecoderWithEmbeddingDataset(embeddings, strings, token2id, max_len=max_len)
    loader  = DataLoader(
        dataset, batch_size=batch_size, shuffle=True,
        num_workers=2, pin_memory=True, persistent_workers=True,
    )

    optimizer = torch.optim.AdamW(decoder.parameters(), lr=lr, weight_decay=1e-2, betas=(0.9, 0.98))

    # label smoothing reduces overconfidence and helps escape plateaus
    criterion = nn.CrossEntropyLoss(ignore_index=-100, label_smoothing=0.1)

    total_steps   = epochs * len(loader)
    warmup_steps  = warmup_epochs * len(loader)

    def lr_lambda(step):
        if step < warmup_steps:
            return step / max(warmup_steps, 1)
        progress = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
        return 0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * progress))  # decays to 10% of lr

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    scaler    = torch.amp.GradScaler('cuda')

    best_loss = float("inf")

    for epoch in range(epochs):
        decoder.train()
        total_loss = 0.0

        pbar = tqdm(loader, desc=f"Epoch {epoch+1}/{epochs}")
        for z_batch, input_ids, labels in pbar:
            z_batch   = z_batch.to(device, non_blocking=True)
            input_ids = input_ids.to(device, non_blocking=True)
            labels    = labels.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast('cuda'):
                logits = decoder(z_batch, target_tokens=input_ids)
                loss   = criterion(logits.reshape(-1, decoder.vocab_size), labels.reshape(-1))

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(decoder.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()   # step every batch, not every epoch

            total_loss += loss.item()
            pbar.set_postfix(loss=f"{loss.item():.4f}", lr=f"{scheduler.get_last_lr()[0]:.2e}")

        avg_loss = total_loss / len(loader)
        print(f"Epoch {epoch+1} — Avg Loss: {avg_loss:.4f}")

        if avg_loss < best_loss:
            best_loss = avg_loss
            torch.save(decoder.state_dict(), save_path)
            print(f"  ✓ saved (best loss: {best_loss:.4f})")


# ─────────────────────────────────────────────────────────────────────────────
# Quick sanity check — run this before training
# ─────────────────────────────────────────────────────────────────────────────

def sanity_check(decoder, token2id, device="cuda"):
    vocab = len(token2id)
    print(f"Vocab size       : {vocab}")
    print(f"Model vocab_size : {decoder.vocab_size}")
    assert vocab == decoder.vocab_size, "MISMATCH — fix vocab_size in constructor"

    z   = torch.randn(4, decoder.z_proj[0].in_features).to(device)
    mem = decoder._build_memory(z)
    print(f"Memory shape     : {mem.shape}")   # (4, z_seq_len+4, d_model)

    inp = torch.randint(0, vocab, (4, 10)).to(device)
    out = decoder(z, inp)
    print(f"Logits shape     : {out.shape}")   # (4, 10, vocab_size)

    random_loss = math.log(vocab)
    print(f"Random-chance loss would be: {random_loss:.3f}")
    print("If epoch-1 loss is already near your stuck value, z_proj or vocab is broken.")
    print("Sanity check passed ✓")

In [44]:
deocder = TransformerJEPADecoder(vocab_size=VOCAB_SIZE, max_len=MAX_LEN)
deocder.to(device)
sanity_check(deocder, TOKEN2ID, device=device)

Vocab size       : 27
Model vocab_size : 27
Memory shape     : torch.Size([4, 12, 256])
Logits shape     : torch.Size([4, 10, 27])
Random-chance loss would be: 3.296
If epoch-1 loss is already near your stuck value, z_proj or vocab is broken.
Sanity check passed ✓


In [45]:
# check your actual max equation length
train_decoder(
    deocder, # Corrected variable name from 'decoder' to 'deocder'
    embeddings,
    equations, # Changed 'strings' (DataFrame) to 'equations' (list of strings)
    token2id,
    PAD_ID, # Corrected variable name from 'pad_id' to 'PAD_ID'
    20, # Corrected variable name from 'max_len' to 'MAX_LEN'
    epochs     = 50,
    batch_size = 256,
    lr         = 3e-4,
    warmup_epochs = 5,
    save_path  = "/content/drive/MyDrive/symbolic_data/final_jepa_model/jepa_decoder_3_march.pt",
)

Epoch 1/50: 100%|██████████| 196/196 [00:11<00:00, 17.68it/s, loss=2.5358, lr=6.00e-05]


Epoch 1 — Avg Loss: 5.8713
  ✓ saved (best loss: 5.8713)


Epoch 2/50: 100%|██████████| 196/196 [00:10<00:00, 18.19it/s, loss=1.9405, lr=1.20e-04]


Epoch 2 — Avg Loss: 2.1539
  ✓ saved (best loss: 2.1539)


Epoch 3/50: 100%|██████████| 196/196 [00:09<00:00, 21.32it/s, loss=1.8113, lr=1.80e-04]


...

Epoch 48 — Avg Loss: 1.1630
  ✓ saved (best loss: 1.1630)


Epoch 49/50: 100%|██████████| 196/196 [00:10<00:00, 18.98it/s, loss=1.1988, lr=3.03e-05]


Epoch 49 — Avg Loss: 1.1614
  ✓ saved (best loss: 1.1614)


Epoch 50/50: 100%|██████████| 196/196 [00:11<00:00, 17.53it/s, loss=1.1720, lr=3.00e-05]


Epoch 50 — Avg Loss: 1.1600
  ✓ saved (best loss: 1.1600)


In [46]:
# ── paste and run this entire cell ──

# Instantiate dataset and loader for evaluation in this cell
dataset = DecoderWithEmbeddingDataset(embeddings, equations, TOKEN2ID, max_len=MAX_LEN)
loader  = DataLoader(
    dataset, batch_size=256, shuffle=True,
    num_workers=2, pin_memory=True, persistent_workers=True,
)

deocder.eval()
z, inp, lbl = next(iter(loader))
z, inp, lbl = z.to(device), inp.to(device), lbl.to(device)

# 1. embedding stats
print("=== Embedding stats ===")
raw = torch.tensor(embeddings).float()
print(f"  raw   mean={raw.mean():.3f}  std={raw.std():.3f}  min={raw.min():.3f}  max={raw.max():.3f}")

# 2. vocab check
print(f"\n=== Vocab ===")
print(f"  len(TOKEN2ID)      = {len(TOKEN2ID)}")
print(f"  decoder.vocab_size = {deocder.vocab_size}") # Corrected variable name from 'decoder' to 'deocder'
print(f"  random-chance loss = {math.log(len(TOKEN2ID)):.3f}")

# 3. what is the model predicting
with torch.no_grad():
    logits = deocder(z, inp) # Corrected variable name from 'decoder' to 'deocder'
preds = logits.argmax(-1)

print(f"\n=== Predictions (first 5) ===")
for i in range(5):
    pred = [ID2TOKEN.get(t.item(), '?') for t in preds[i]]
    true = [ID2TOKEN.get(t.item(), '?') for t in lbl[i] if t.item() != -100]
    print(f"  TRUE: {' '.join(true)}")
    print(f"  PRED: {' '.join(pred)}")
    print()

# 4. is it predicting the same token over and over?
print("=== Most predicted tokens ===")
counts = torch.bincount(preds.flatten(), minlength=deocder.vocab_size).float() # Corrected variable name from 'decoder' to 'deocder'
top5   = counts.topk(5)
for val, idx in zip(top5.values, top5.indices):
    print(f"  '{ID2TOKEN.get(idx.item(), '?')}' predicted {int(val)} times ({100*val/preds.numel():.1f}%) ")

# 5. per-position accuracy
mask    = (lbl != -100)
correct = (preds == lbl) & mask
print(f"\n=== Per-position accuracy: {100*correct.float().sum()/mask.float().sum():.1f}% ===")

per_pos = correct.float().mean(0)
print("  By position:", [f"{x:.2f}" for x in per_pos[:10].tolist()])

=== Embedding stats ===
  raw   mean=0.041  std=0.488  min=-1.793  max=1.779

=== Vocab ===
  len(TOKEN2ID)      = 27
  decoder.vocab_size = 27
  random-chance loss = 3.296

=== Predictions (first 5) ===
  TRUE: ADD DIV <UNK> sin <UNK> DIV x2 <C_0> <EOS>
  PRED: DIV DIV <UNK> sin <UNK> DIV x2 <C_0> <EOS> <EOS> <EOS> <EOS> <EOS> <EOS> <EOS> <EOS> <EOS> <EOS> <EOS> <EOS> <EOS> <EOS> <EOS> <EOS> <EOS> <EOS> <EOS> <EOS> <EOS>

  TRUE: ADD MUL MUL <UNK> x1 MUL x1 x1 exp <C_0> <EOS>
  PRED: ADD MUL MUL <UNK> x1 MUL x1 x1 exp <C_0> <EOS> <EOS> <EOS> <EOS> <EOS> <EOS> <EOS> <EOS> <EOS> <EOS> <EOS> <EOS> <EOS> <EOS> <EOS> <EOS> <EOS> <EOS> <EOS>

  TRUE: SUB MUL x2 ADD MUL sin x2 SUB <UNK> x2 POW <UNK> x1 DIV MUL x2 MUL x1 <C_0> SUB sin MUL exp <C_1> MUL x1 <C_2> DIV x1
  PRED: SUB MUL SUB MUL SUB SUB x2 SUB <UNK> x2 MUL <UNK> x1 MUL MUL MUL MUL x1 <C_0> SUB <C_1> x2 <C_1> <C_1> MUL x2 x2 MUL x2

  TRUE: ADD ADD cos <C_0> MUL <C_1> x1 <C_2> <EOS>
  PRED: ADD ADD cos <C_0> MUL <C_1> x1 <C_2> <EO